# ML-2 screening
Train the screening priority model on synthetic applications.

In [1]:
from pathlib import Path
import json
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split

base = Path.cwd()
data = pd.read_csv(base / '../data/ml2_applications.csv')
raw_features = [c for c in data.columns if c != 'passed']
categorical = ['sex', 'study_type', 'phase']
numeric = [c for c in raw_features if c not in categorical]
encoded = pd.get_dummies(data[raw_features], columns=categorical)
feature_columns = list(encoded.columns)
X_train, X_test, y_train, y_test = train_test_split(encoded, data['passed'], test_size=0.3, stratify=data['passed'], random_state=42)
model = RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=20, random_state=42)
model.fit(X_train, y_train)
baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
test_probability = model.predict_proba(X_test)[:, list(model.classes_).index(1)]
print('train accuracy:', accuracy_score(y_train, model.predict(X_train)))
print('test accuracy:', accuracy_score(y_test, model.predict(X_test)))
print('ROC AUC:', roc_auc_score(y_test, test_probability))
print('majority baseline:', accuracy_score(y_test, baseline.predict(X_test)))
print('confusion matrix:\n', confusion_matrix(y_test, model.predict(X_test)))
for label, mask in [('High', test_probability >= .70), ('Medium', (test_probability >= .40) & (test_probability < .70)), ('Low', test_probability < .40)]:
    print(label, 'pass rate:', y_test[mask].mean() if mask.any() else 0)
metrics = {'train_accuracy': accuracy_score(y_train, model.predict(X_train)), 'test_accuracy': accuracy_score(y_test, model.predict(X_test)), 'roc_auc': roc_auc_score(y_test, test_probability), 'majority_baseline_accuracy': accuracy_score(y_test, baseline.predict(X_test))}
models = base / '../models'
import joblib
joblib.dump(model, models / 'ml2_screening.joblib')
with (models / 'ml2_screening_meta.json').open('w', encoding='utf-8') as file:
    json.dump({'model_name': 'ml2', 'version': 'ml2-v1.0', 'trained_at': pd.Timestamp.now().isoformat(), 'algorithm': 'RandomForestClassifier', 'params': {'n_estimators': 200, 'max_depth': 8, 'min_samples_leaf': 20, 'random_state': 42}, 'raw_features': raw_features, 'feature_columns': feature_columns, 'positive_class': 1, 'metrics': metrics, 'n_train': len(X_train), 'n_test': len(X_test)}, file, indent=2)


train accuracy: 0.7316071428571429
test accuracy: 0.6604166666666667
ROC AUC: 0.7206728207268145
majority baseline: 0.5608333333333333
confusion matrix:
 [[ 537  517]
 [ 298 1048]]
High pass rate: 0.8188775510204082
Medium pass rate: 0.5563463819691578
Low pass rate: 0.2701863354037267
